# Cadrage du Problème

L'objectif est un **classification binaire** qui indique si un vol sera en retard ou non. Retard = ArrDelay > 15 minutes

In [ ]:
# Importation des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Métriques
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, roc_auc_score


# Chargement des données

In [ ]:
file_path = 'flight-delay-dataset-20182022/Combined_Flights_2021.parquet'

df = pd.read_parquet(file_path)

print(f"Dataset complet chargé : {len(df):,} vols")

## Exploration des données (EDA)


In [ ]:
# Échantillon pour exploration initiale rapide (développement uniquement)
df_sample = df.sample(n=100000, random_state=42).reset_index(drop=True)
df_sample.head()

In [ ]:
df_sample.describe()

In [ ]:
df_sample.info()

On observe des données manquantes dans plusieurs colonnes, principalement liées aux informations de vol effectives (retards, temps de taxi, etc.), ce qui est cohérent avec la présence de vols annulés ou détournés :

- **Départ** : `DepTime`, `DepDelay`, `DepDel15`, etc. (~1,8% de manquants).
- **Arrivée** : `ArrTime`, `ArrDelay`, `ArrDel15`, etc. (~2,1% de manquants).
- **Détails techniques** : `Tail_Number` (384 manquants) et `TaxiOut`/`TaxiIn`.
- **Informations de vol** : `AirTime` et `ActualElapsedTime`.

Ces valeurs devront être traitées (imputation ou suppression) avant l'entraînement du modèle de classification.

Notre objectif étant de faire une classification binaire sur `ArrDel15`, nous devons supprimer les lignes où cette variable est manquante (environ 2,1% des données). Il s'agit des vols annulés.

Les valeurs de temps (ex. `DepTime`, `ArrTime`) sont au format float (ex. 1345.0 pour 13h45), ce qui peut nécessiter une conversion en format horaire standard pour une meilleure interprétation.

`DayOfWeek` est codé de 1 (lundi) à 7 (dimanche), ce qui est utile pour capturer les variations hebdomadaires des retards.
`DayofMonth` et `Month` sont également présents, permettant d'analyser les tendances saisonnières. Ils sont codés avec des entiers.

D'après l'exploration des données, nous pouvons classifier les variables ainsi :

**Variables Catégorielles :**
*   **Identifiants & Codes :** `Airline`, `Origin`, `Dest`, `Marketing_Airline_Network`, `Operating_Airline`, `Tail_Number`, `IATA_Code_Marketing_Airline`, etc.
*   **Temporelles (discrètes) :** `Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek`.
*   **Indicateurs binaires :** `Cancelled`, `Diverted`, `DepDel15`, `ArrDel15` (Target).
*   **Groupements :** `DepartureDelayGroups`, `ArrivalDelayGroups`, `DistanceGroup`, `DepTimeBlk`, `ArrTimeBlk`.

**Variables Continues (Numériques) :**
*   **Temps de vol & Retards :** `DepDelay`, `DepDelayMinutes`, `ArrDelay`, `ArrDelayMinutes`, `AirTime`, `ActualElapsedTime`, `CRSElapsedTime`, `TaxiIn`, `TaxiOut`.
*   **Distance :** `Distance`.
*   **Horaires (à convertir) :** `DepTime`, `ArrTime`, `CRSDepTime`, `CRSArrTime`, `WheelsOff`, `WheelsOn`.

**Note :** Certaines variables comme `OriginAirportID` ou `OriginStateFips` sont stockées comme des entiers (`int64`) mais sont conceptuellement des variables **catégorielles** (identifiants).


### Nettoyage des données

In [ ]:
df_clean = df[(df['Cancelled'] == False) & (df['Diverted'] == False)].copy()

# Supprimer les lignes avec ArrDel15 manquant (vols sans info de retard)
df_clean = df_clean.dropna(subset=['ArrDel15'])

print(f"Dataset nettoyé : {len(df_clean):,} vols")

### Feature Engineering

In [ ]:
df_clean['DepHour'] = df_clean['CRSDepTime'] // 100
df_clean['IsWeekend'] = (df_clean['DayOfWeek'] >= 6).astype(int)
df_clean['IsHolidayMonth'] = df_clean['Month'].isin([6, 7, 12]).astype(int)

### Séparation Train/Test - Protection contre le Data Snooping

Toutes les analyses EDA suivantes doivent être faites sur `df_train` uniquement.

In [ ]:
from sklearn.model_selection import train_test_split

# Créer le jeu de test (20%) et le mettre de côté
df_train, df_test = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean['ArrDel15']  # Préserve la proportion des classes
)

print(f"Training set: {len(df_train)} samples")
print(f"Test set: {len(df_test)} samples")
print(f"\nTest set - Distribution ArrDel15:\n{df_test['ArrDel15'].value_counts(normalize=True)}")

# ⚠️ NE PLUS TOUCHER df_test jusqu'à l'évaluation finale

## Préparation des données pour le ML (Preprocessing)

In [ ]:
# Définir les features et la target
features = [
    # Temporelles
    'Month', 'DayOfWeek', 'DayofMonth', 'IsWeekend', 'IsHolidayMonth',
    # Horaire
    'CRSDepTime', 'DepHour',
    # Catégorielles
    'Marketing_Airline_Network', 'Airline', 'Origin', 'Dest',
    # Distance
    'Distance'
]
target = 'ArrDel15'

# Séparation X/y pour train et test
X_train = df_train[features].copy()
y_train = df_train[target].copy()

X_test = df_test[features].copy()
y_test = df_test[target].copy()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\n✅ {len(features)} features sélectionnées")

In [ ]:
# Identification des colonnes numériques et catégorielles
categorical_features = ['Marketing_Airline_Network', 'Airline', 'Origin', 'Dest']
numerical_features = ['Month', 'DayOfWeek', 'DayofMonth', 'IsWeekend', 'IsHolidayMonth', 
                      'CRSDepTime', 'DepHour', 'Distance']

# Pipeline pour les features numériques : imputation + standardisation
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline pour les features catégorielles : imputation + encodage one-hot
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer pour appliquer les pipelines
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

print(f"\nFeatures numériques ({len(numerical_features)}): {numerical_features}")
print(f"Features catégorielles ({len(categorical_features)}): {categorical_features}")
print(f"Total features: {len(numerical_features) + len(categorical_features)}")

### Exploration des données (EDA) - Suite

In [ ]:
print("Distribution de la variable cible (ArrDel15) :")
print(df_train[target].value_counts(normalize=True))
plt.figure(figsize=(6, 4))
sns.countplot(x=target, data=df_train, hue=target, palette='viridis', legend=False)
plt.title("Répartition des retards (0 = À l'heure, 1 = Retard > 15min)")
plt.show()

In [ ]:
airline_delay = df_train.groupby('Airline')[target].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(y=airline_delay.index, x=airline_delay.values, hue=airline_delay.index, palette='coolwarm', legend=False)
plt.title('Taux de retard moyen par Compagnie Aérienne')
plt.xlabel('Proportion de retards')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x=target, y='Distance', data=df_train, hue=target, palette=['green', 'red'], legend=False)
plt.xticks([0, 1], ['À l\'heure', 'En retard'])
plt.title('Distribution de la distance par statut de retard')
plt.xlabel('Statut du vol')
plt.ylabel('Distance (miles)')
plt.show()

In [ ]:
# Matrice de corrélation étendue avec features temporelles et opérationnelles
corr_features = [
    # Temporel
    'Month', 'Quarter', 'DayOfWeek', 'DayofMonth', 'IsWeekend', 'IsHolidayMonth',
    # Horaire
    'CRSDepTime', 'DepHour',
    # Distance/Durée
    'Distance', 'CRSElapsedTime', 'DistanceGroup',
    # Target
    target
]

corr_data = df_train[corr_features].dropna()

plt.figure(figsize=(14, 10))
correlation_matrix = corr_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            linewidths=0.5, cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de corrélation étendue - Features temporelles et opérationnelles', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# Affichage des corrélations avec la target triées
print("\nCorrélations avec ArrDel15 (triées par importance) :")
target_corr = correlation_matrix[target].drop(target).sort_values(ascending=False)
print(target_corr)

In [ ]:
# Relation entre distance et retard
plt.figure(figsize=(10, 6))
distance_delay = df_train.groupby('DistanceGroup')[target].mean().sort_index()
sns.lineplot(x=distance_delay.index, y=distance_delay.values, marker='o', color='coral', linewidth=2)
plt.title('Taux de retard en fonction du groupe de distance')
plt.xlabel('Groupe de distance')
plt.ylabel('Proportion de retards')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Distribution des retards par plage horaire de départ
plt.figure(figsize=(12, 6))
delay_by_time = df_train.groupby('DepTimeBlk')[target].mean().sort_values(ascending=False)
sns.barplot(x=delay_by_time.index, y=delay_by_time.values, hue=delay_by_time.index, palette='viridis', legend=False)
plt.xticks(rotation=45, ha='right')
plt.title('Taux de retard par plage horaire de départ')
plt.xlabel('Plage horaire')
plt.ylabel('Proportion de retards')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des retards par mois
plt.figure(figsize=(10, 5))
month_labels = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun', 'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
delay_by_month = df_train.groupby('Month')[target].mean().sort_index()
sns.barplot(x=delay_by_month.index, y=delay_by_month.values, palette='Oranges_d', hue=delay_by_month.index, legend=False)
plt.xticks(range(12), month_labels)
plt.title('Taux de retard par mois')
plt.xlabel('Mois')
plt.ylabel('Proportion de retards')
plt.show()